In [2]:
import os
from openai import OpenAI
import cohere
from dotenv import load_dotenv


load_dotenv() 
api_key = os.getenv("OPENAI_API_KEY")
api_key_co = os.getenv("COHERE_API_KEY")

os.environ['OPENAI_API_KEY'] = api_key
os.environ['COHERE_API_KEY'] = api_key_co

In [3]:
co = cohere.ClientV2(
    api_key_co
)

In [4]:
client = OpenAI()

## zero-shot prompting

In [6]:
def zero_shot_prompt(prompt):
    """Demonstrate zero-shot prompting."""

    response = co.chat(
        model="command-a-03-2025",
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    print("Zero-shot Prompting:\n", response.message.content[0].text)

zero_shot_prompt("Explain the concept of quantum computing in simple terms.")

Zero-shot Prompting:
 Quantum computing is a new way of processing information that uses the strange and powerful rules of quantum mechanics, the science that explains how the tiniest particles in the universe behave. Here’s a simple breakdown:

1. **Classical vs. Quantum Bits (Qubits):**
   - In regular computers, information is stored in **bits**, which can be either a **0** or a **1**.
   - In quantum computers, information is stored in **qubits** (quantum bits). Qubits can be **0**, **1**, or both at the same time, thanks to a quantum property called **superposition**.

2. **Superposition:**
   - Imagine a coin spinning in the air. While it’s spinning, it’s neither fully heads nor fully tails—it’s in a mix of both states. Qubits work similarly; they can exist in multiple states simultaneously.
   - This allows quantum computers to process a vast number of possibilities at once, making them incredibly powerful for certain tasks.

3. **Entanglement:**
   - Another quantum property is

## one-shot prompting

In [7]:
def few_shot_prompt(prompt):
    """Demonstrate few-shot prompting."""

    response = co.chat(
        model="command-a-03-2025",
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    print("Few-shot Prompting:\n", response.message.content[0].text)

prompt = (
        "Translate the following English phrases to French:\n"
        "English: Hello, how are you?\n"
        "French: Bonjour, comment ça va?\n"
        "English: What is your name?\n"
        "French: Comment vous appelez-vous?\n"
        "English: Where is the library?\n"
        "French:")

few_shot_prompt(prompt)

Few-shot Prompting:
 Où est la bibliothèque ?


## prompt chaining

In [8]:
def chain_part1(initial_prompt):
    """Demonstrate prompt chaining."""
    # First prompt
    response1 = co.chat(
        model="command-a-03-2025",
        messages=[
          {"role": "user", "content": initial_prompt}
        ]
    )
    languages = response1.message.content[0].text

    return languages

def chain_part2(follow_up_prompt):
    # Second prompt using the output of the first
    response2 = co.chat(
        model="command-a-03-2025",
        messages=[
          {"role": "user", "content": follow_up_prompt}
        ]
    )
    return response2.message.content[0].text

initial_prompt = "Provide a list of three popular programming languages."

response_part1 = chain_part1(initial_prompt)

follow_up_prompt = f"Explain the primary use case for each of these programming languages:\n{response_part1}"

response_part2 = chain_part2(follow_up_prompt)

print("Part 1:\n", response_part1,"\n")
print("-"*150,"\n")
print("Part 2:\n", response_part2)

Part 1:
 Here are three popular programming languages:

1. **Python**: Known for its simplicity and readability, Python is widely used in web development, data analysis, artificial intelligence, machine learning, and scientific computing. Its extensive libraries and frameworks, such as Django, Flask, TensorFlow, and Pandas, make it a versatile choice for various applications.

2. **JavaScript**: The primary language for web development, JavaScript is essential for creating interactive and dynamic web pages. It is supported by all major web browsers and is also used in server-side development through platforms like Node.js. JavaScript's popularity has grown with the rise of single-page applications and frameworks like React, Angular, and Vue.js.

3. **Java**: A robust, object-oriented language, Java is widely used for enterprise-level applications, Android mobile app development, and large-scale systems. Its "write once, run anywhere" (WORA) capability, thanks to the Java Virtual Machin

## Chain of Thought (CoT)

In [9]:
def chain_of_thought(prompt):
    """Demonstrate chain-of-thought reasoning."""

    response = co.chat(
        model="command-a-03-2025",
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    print("Chain of Thought:\n", response.message.content[0].text)

prompt = (
        "A train leaves the station at 60 miles per hour. Another train leaves the same station "
        "one hour later at 80 miles per hour. How long will it take for the second train to catch the first train? "
        "Explain your reasoning step by step."
    )

chain_of_thought(prompt)

Chain of Thought:
 To determine how long it will take for the second train to catch the first train, let's break down the problem step by step.

### **Step 1: Define the Variables**
- Let \( t \) be the time (in hours) it takes for the second train to catch the first train after the second train departs.

### **Step 2: Calculate the Distance Traveled by Each Train**
- **First Train**: 
  - Speed = 60 miles per hour.
  - Time = \( t + 1 \) hours (since it leaves 1 hour earlier).
  - Distance = Speed × Time = \( 60 \times (t + 1) \).

- **Second Train**:
  - Speed = 80 miles per hour.
  - Time = \( t \) hours.
  - Distance = Speed × Time = \( 80 \times t \).

### **Step 3: Set Up the Equation**
When the second train catches the first train, both trains will have traveled the same distance. Therefore:
\[
60 \times (t + 1) = 80 \times t
\]

### **Step 4: Solve for \( t \)**
Expand and simplify the equation:
\[
60t + 60 = 80t
\]
Subtract \( 60t \) from both sides:
\[
60 = 20t
\]
Divide both